<a href="https://colab.research.google.com/github/amogh-4050/pesuio_agentic_systems101/blob/pdf_implementation/Copy_of_advanced_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Replacements
**FAISS instead of Qdrant for vector storage**

**Sentence Transformers instead of llama-index embeddings**

**Groq instead of OpenAI**

**Unstructured library for document parsing**

**Using sentence-based chunking**

**ReRanking rag technique**

##Setting up the Environment

In [ ]:
!pip install openai
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf2
!pip install python-dotenv
!pip install nltk
!pip install gradio
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2
  Attempting uninstall: huggingface-hub

In [ ]:
# Standard library imports
import os
import openai
import numpy as np
import faiss
import PyPDF2
import nltk

# Third-party imports
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import tempfile
import shutil
from typing import List, Dict
import groq

# Download necessary NLTK data
nltk.download('punkt')

# Load environment variables
load_dotenv()

# Set Groq API key
groq_api_key = "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"
groq_client = groq.Groq(api_key=groq_api_key)

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


##Creating a document processor

In [ ]:
#Sentence-Based Chunking:
#This method involves breaking down the text into individual sentences while preserving sentence boundaries.
class DocumentProcessor:
    def __init__(self, chunk_size: int = 1000):
        self.chunk_size = chunk_size

    def read_pdf(self, file_path: str) -> str:
        #Read PDF file and return text content
        with open(file_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            text = ''
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:  # Check if text extraction was successful
                    text += page_text + '\n'
        return text.strip()

    def chunk_text(self, text: str) -> List[str]:
        #Split text into chunks using sentence tokenization
        sentences = sent_tokenize(text)
        chunks = []
        current_chunk = []

        for sentence in sentences:
            # Check if adding this sentence would exceed the chunk size
            if sum(len(s) for s in current_chunk) + len(sentence) < self.chunk_size:
                current_chunk.append(sentence)
            else:
                # If current chunk is full, append it to chunks and start a new one
                chunks.append(' '.join(current_chunk).strip())
                current_chunk = [sentence]  # Start new chunk with the current sentence

        # Add any remaining sentences as a final chunk
        if current_chunk:
            chunks.append(' '.join(current_chunk).strip())

        return chunks

##Making a vector database

In [ ]:
#FAISS (Facebook AI Similarity Search) Database
#faiss db for efficient similarity search among vector embeddings.
class VectorStore:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.dimension = 384  # Dimension of the chosen model's embeddings
        self.index = faiss.IndexFlatL2(self.dimension)
        self.texts = []

    def add_texts(self, texts: List[str]):
        #Add texts to the vector store.
        embeddings = self.model.encode(texts)
        self.index.add(np.array(embeddings).astype('float32'))
        self.texts.extend(texts)

    def similarity_search(self, query: str, k: int = 3) -> List[str]:
        #Search for similar texts using the query.
        query_embedding = self.model.encode([query])
        distances, indices = self.index.search(
            np.array(query_embedding).astype('float32'), k
        )
        return [self.texts[i] for i in indices[0]]

##Creating the RAGchat bot

In [ ]:
#Bi-Encoder ReRanking Techniques
#ReRanking involves refining the results obtained from initial retrieval by applying additional scoring mechanisms to ensure that the most relevant documents are presented first.
class RAGChat:
    def __init__(self):
        self.doc_processor = DocumentProcessor()
        self.vector_store = VectorStore()
        self.chat_history = []

    def process_document(self, file_path: str) -> str:
        #Process a document and add it to the vector store

        try:
            text = self.doc_processor.read_pdf(file_path)
            chunks = self.doc_processor.chunk_text(text)
            self.vector_store.add_texts(chunks)
            return f"Successfully processed document with {len(chunks)} chunks."
        except Exception as e:
            return f"Error processing document: {str(e)}"

    def rerank_documents(self, query: str, context_chunks: List[str]) -> List[str]:
        #Re-rank the retrieved context chunks based on their relevance to the query

        # Embed the query
        query_embedding = self.vector_store.model.encode(query)

        # Embed the context chunks
        context_embeddings = self.vector_store.model.encode(context_chunks)

        # Calculate cosine similarities
        similarities = np.dot(context_embeddings, query_embedding) / (
            np.linalg.norm(context_embeddings, axis=1) * np.linalg.norm(query_embedding))

        # Sort indices by similarity score
        ranked_indices = np.argsort(similarities)[::-1]

        # Return ranked context chunks
        return [context_chunks[i] for i in ranked_indices]

    def generate_response(self, query: str) -> (str, str):
        #Generate a response using RAG with re-ranking

        # Retrieve relevant context chunks for the query
        context_chunks = self.vector_store.similarity_search(query)

        if not context_chunks:
            return "No relevant context found.", ""

        # Re-rank the retrieved context chunks
        ranked_context_chunks = self.rerank_documents(query, context_chunks)

        # Get the complete chunk(s) used for generating the answer
        complete_context = "\n\n".join(ranked_context_chunks[:5])  # Join top 5 ranked chunks if necessary

        messages = [
            {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided context."},
            {"role": "user", "content": f"Context: {complete_context} Question: {query} Answer:"}
        ]

        try:
            completion = groq_client.chat.completions.create(
                messages=messages,
                #"mixtral-8x7b-32768" - gives unnecessarily large answers. Sometimes answers are not provided with he correct context
                #llama-3.2-90b-vision-preview" - good but gives respose based on content not mentioned in the pdf
                #"llama-3.1-70b-versatile" - best gives accurate response based on the contents of the pdf
                model="llama-3.1-70b-versatile",  # Use a model that gives relevant answers
                temperature=0.1,
                max_tokens=1000,
            )
            answer = completion.choices[0].message.content
            self.chat_history.append({"query": query, "response": answer})
            return answer, complete_context  # Return both answer and complete context
        except Exception as e:
            return f"Error generating response: {str(e)}", ""

##Gradio Application

In [ ]:
import gradio as gr

def create_demo():
    rag_chat = RAGChat()  # Initialize the RAGChat instance

    def process_file(file):
        return rag_chat.process_document(file.name)  # Process the uploaded PDF file

    def chat(message, history):
        # Check for general greetings or small talk
        if message.lower() in ["hi", "hello", "hey", "how are you?", "what's up?", "help"]:
            response = "Hello! How can I assist you today? Please upload a PDF document if you have one."
            history.append((message, response))  # Append user message and response
            return "", history, ""  # Clear input box and return updated history

        # Check if any documents have been processed before answering questions
        if not rag_chat.vector_store.texts:
            response = "Please upload a PDF document before asking questions."
            history.append((message, response))  # Append user message and response
            return message, history, ""  # Return original message and updated history, clear context display

        # Generate response using the RAGChat instance
        response, complete_context = rag_chat.generate_response(message)
        history.append((message, response))  # Append user message and response
        return "", history, complete_context  # Clear input box after submitting and return complete context

    # Create Gradio interface
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown('## Simple RAG Chatbot\nUpload a PDF document and ask questions about its content.')

        with gr.Row():
            with gr.Column(scale=1):
                file_input = gr.File(
                    label="Upload PDF Document",
                    file_types=[".pdf"]
                )
                upload_button = gr.Button("Process Document")
                status_box = gr.Textbox(label="Status", interactive=False)

            with gr.Column(scale=2):
                chatbot = gr.Chatbot(label="Chat History")
                msg = gr.Textbox(
                    label="Your Message",
                    placeholder="Ask a question about the document..."
                )
                clear = gr.Button("Clear Chat")
                context_display = gr.Textbox(label="Relevant Context Chunk", interactive=False)

        # Set up event handlers
        upload_button.click(process_file, inputs=[file_input], outputs=[status_box])
        msg.submit(chat, inputs=[msg, chatbot], outputs=[msg, chatbot, context_display])  # Updated outputs to include context_display
        clear.click(lambda: None, None, chatbot, queue=False)

    return demo

if __name__ == "__main__":
    demo = create_demo()
    demo.launch(share=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/gradio/components/chatbot.py:223: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ffaa8d0afdb123952.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
